In [11]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import pickle
import json

In [15]:
df = pd.read_csv("Final_Augmented_dataset_Diseases_and_Symptoms.csv", header=0, index_col=0)

print(f" Loaded {df.shape[0]:,} rows.")

counts = df.index.value_counts()
valid_diseases = counts[counts >= 5].index
df = df[df.index.isin(valid_diseases)].copy()

print(f" Filtered data: {df.shape[0]:,} rows remaining.")
print(f" Unique Diseases remaining: {df.index.nunique()}")

 Loaded 246,945 rows.
 Filtered data: 246,823 rows remaining.
 Unique Diseases remaining: 721


In [19]:
X = df.values.astype(np.int8)   # int8 saves RAM
y_raw = df.index.tolist()      # Disease names from the index

# Fit encoder ONLY on the filtered diseases
le = LabelEncoder()
y = le.fit_transform(y_raw)

# Save for app.py
symptom_names = df.columns.tolist()
with open("symptom_names.json", "w") as f:
    json.dump(symptom_names, f)
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

print(f" {len(symptom_names)} symptoms saved")
print(f" {len(le.classes_)} disease classes saved ")

 377 symptoms saved
 721 disease classes saved 


In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y           # This now works because we filtered classes < 5
)
print(f"\n Train : {len(X_train):,} rows")
print(f" Test  : {len(X_test):,} rows")


 Train : 197,458 rows
 Test  : 49,365 rows


In [23]:
model = RandomForestClassifier(
    n_estimators=300,        
    max_depth=30,            
    min_samples_split=10,    
    min_samples_leaf=2,      
    max_features="sqrt",     
    class_weight="balanced", 
    n_jobs=-1,               
    random_state=42,
    verbose=1                
)

model.fit(X_train, y_train)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:   24.5s
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:  2.1min
[Parallel(n_jobs=-1)]: Done 300 out of 300 | elapsed:  3.5min finished


RandomForestClassifier(class_weight='balanced', max_depth=30,
                       min_samples_leaf=2, min_samples_split=10,
                       n_estimators=300, n_jobs=-1, random_state=42, verbose=1)

In [25]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"  Overall Accuracy : {accuracy * 100:.2f}%")

# The target_names will now perfectly match the length of classes in y_test
print("\n Per-disease breakdown (first 20 diseases):")
report = classification_report(
    y_test, y_pred,
    target_names=le.classes_,
    output_dict=True
)
report_df = pd.DataFrame(report).T
print(report_df[["precision","recall","f1-score","support"]]
      .iloc[:20].round(2).to_string())

[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    4.1s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:   22.4s
[Parallel(n_jobs=12)]: Done 300 out of 300 | elapsed:   35.6s finished


  Overall Accuracy : 80.56%

 Per-disease breakdown (first 20 diseases):
                                            precision  recall  f1-score  support
abdominal aortic aneurysm                        0.96    0.93      0.95     28.0
abdominal hernia                                 0.93    0.98      0.95     81.0
abscess of nose                                  0.88    0.91      0.90     58.0
abscess of the lung                              1.00    1.00      1.00      4.0
abscess of the pharynx                           0.98    0.85      0.91     68.0
acanthosis nigricans                             0.10    0.83      0.19      6.0
acariasis                                        1.00    0.86      0.92      7.0
achalasia                                        0.75    0.71      0.73     17.0
acne                                             0.91    0.64      0.75     99.0
actinic keratosis                                0.88    0.68      0.77    182.0
acute bronchiolitis                 

C:\Users\Probook\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Probook\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Probook\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [29]:
print("\nTop 20 most important symptoms for diagnosis:")
importance = pd.Series(model.feature_importances_, index=symptom_names)
top20 = importance.nlargest(20)
for sym, score in top20.items():
    bar = "█" * int(score * 500)
    print(f"  {sym:<35} {bar} {score:.4f}")

# ── 7. SAVE MODEL ────────────────────────────────────────────
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

print(f" model.pkl saved.")


Top 20 most important symptoms for diagnosis:
  cough                               ██████████ 0.0201
  shortness of breath                 █████████ 0.0185
  headache                            ██████ 0.0139
  sharp abdominal pain                ██████ 0.0133
  sharp chest pain                    ██████ 0.0125
  dizziness                           ██████ 0.0124
  depressive or psychotic symptoms    ██████ 0.0123
  emotional symptoms                  ██████ 0.0121
  vomiting                            █████ 0.0113
  nasal congestion                    █████ 0.0107
  difficulty speaking                 ████ 0.0094
  fatigue                             ████ 0.0091
  hot flashes                         ████ 0.0091
  anxiety and nervousness             ████ 0.0089
  arm stiffness or tightness          ████ 0.0087
  elbow weakness                      ████ 0.0085
  lack of growth                      ████ 0.0085
  nausea                              ████ 0.0081
  diminished hearing        